# Here we fit the position of the Moon in order to get the detectors lines of sight

In [ ]:
# Allows the modification of the files imported without having to restart kernel or re-import them
%load_ext autoreload
%autoreload 2

In [ ]:
%config InlineBackend.figure_format='retina'
from IPython import display
display.display(display.HTML("<style>.container { width:95% !important; }</style>"))

# to have interactive plots : pip install ipympl
%matplotlib ipympl

### General imports
import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
import pickle

from scipy.optimize import least_squares

plt.rc('figure',figsize=(25,10))
plt.rc('font',size=16)

#### QUBIC IMPORT

import pipeline_moon_functions as pmf


In [ ]:
datadir_MoonProject = "/Users/huchet/Documents/code/scripts/MoonProject/"
datadir_maynooth = "/Users/huchet/Documents/code/data/"
dir_plots = "/Users/huchet/qubic/qubic/scripts/MoonProject/figures/"

In [ ]:
name_data = "202607_stack"
nside = 512
zenith = [0, 90, 0]

In [ ]:
# We save the clipped maps!
file_name = "allmaps-{}_nside{}_zenith.pkl".format(name_data, nside)

allTESNum, allmaps, chosen_patch = pickle.load(open( datadir_MoonProject + file_name, "rb" ) )
print("Read {}".format(file_name))

remap = pmf.get_remapping_operator(qubic_patch=chosen_patch, shape_patch=np.shape(chosen_patch), shape_full=hp.nside2npix(nside))

# We load Maynooth detector line of sight simulation as a guess for our fit

In [ ]:
# %%script echo skipping

#### Offsets from Créidhe
quadrant = 3 # for TD, quadrant = 3; formerly "pointing_offsets_fixed_hole.pickle"
# import pickle
offsets = pickle.load( open( datadir_maynooth + 'pointing_offsets_Q{}.pickle'.format(quadrant), 'rb') )
print(np.shape(offsets))

mytesn = np.array(allTESNum)
maynooth_horiz = offsets[mytesn -1 , :]
print(np.shape(maynooth_horiz))

######## Here we apply the inversion betwwen Az and El ##############
invert_azel = True # was due to an inversion in the numbering of TES, or was it? It doesn't fit well with the inversion
if invert_azel is True:
    maynooth_horiz = np.flip(maynooth_horiz, axis=1)
# print(maynooth_horiz[0])

invert_az = True # might be an error in the data taking (orientation east/west)?
if invert_az is True:
    maynooth_horiz[:, 0] = -maynooth_horiz[:, 0]
# print(maynooth_horiz[0])

invert_el = True # might be because we do el - elmoon in our data?
if invert_el is True:
    maynooth_horiz[:, 1] = -maynooth_horiz[:, 1]

allTESNum = np.arange(256) + 1


### We rotate Maynooth to zenith (note that it is in general not the same as fitting at zenith from the start)

In [ ]:
maynooth_cart = pmf.spherical2cartesian(1, maynooth_horiz[:, 0], maynooth_horiz[:, 1], coord="horizontal", axis="last")
rot_mat_zen = pmf.get_simple_rotation_matrix(axis="y", angle=np.radians(90)) # we go back to zenith
maynooth_zen_cart = np.einsum("li,ik->lk", maynooth_cart, rot_mat_zen)
maynooth_zen = pmf.cartesian2spherical(maynooth_zen_cart[:, 0], maynooth_zen_cart[:, 1], maynooth_zen_cart[:, 2], coord="horizontal", axis="last")[:, 1:]


In [ ]:
# %%script echo skipping

# We perform the Moon position fit and show the result for one TES

### Pick a TES number
mynum = 12 #64 #170 #71 #211 #179 #61 #96 #247 # 152
# print("TES {} is {} in visibly_ok_arr".format(mynum, visibly_ok_arr[mynum - 1]))

# myrotinit = np.array([-1, 4, 0]) # np.array([0, 0, 0]) # np.array([-1, -4., 0])
# myrotinit = np.array([0, 0, 0])
myrotinit = np.array([0, 90, 0]) # zenith is better
zenith = np.array([0, 90, 0])
# myrotinit = np.array([-0.5, -4.5, 0])
# myrotinit = np.array([-1, 4])

idx = np.where(np.array(allTESNum)==mynum)[0][0]
#hp.gnomview(allmaps[idx,:], reso=10, rot=myrotinit, title='TES#{}'.format(mynum), min=-5e3, max=2.5e4, sub=(1,4,1))
# reso=4
# xs = 201
# reso=5
xs = 301
reso = 5
# pack = scantype, newazt, newelt, nside
pack = None

# resfit, mapxy, mapfit, extent, fig = pmf.fit_one_tes(allmaps[idx, :], xs, reso, rot=myrotinit, verbose=True, doplot=True,
#                                                     renorm=True, return_images=True, pack=pack)

# print(maynooth_zen[idx])
# resfit, mapxy, mapfit, extent, fig, ijres, ijerr = pmf.fit_one_tes(allmaps[idx, :], xs, reso, rot=myrotinit, return_images=True, renorm=True, doplot=True, azelguess=maynooth_zen[idx], verbose=True, distok=60)

# full_det_pos_zen = np.load('full_det_pos_zen.npy')
# resfit, mapxy, mapfit, extent, fig, ijres, ijerr = pmf.fit_one_tes(allmaps[idx, :], xs, reso, rot=myrotinit, return_images=True, renorm=True, doplot=True, azelguess=full_det_pos_zen[idx], verbose=True, distok=25)

# these positions are from a previous fit with telescope l.o.s. at zenith!
fitted_maynooth = np.load('fitted_maynooth.npy')
resfit, mapxy, mapfit, extent, fig, ijres, ijerr = pmf.fit_one_tes(remap(allmaps[idx, :]), xs, reso, rot=myrotinit, return_images=True, renorm=True, doplot=True, azelguess=fitted_maynooth[idx], verbose=True, distok=25)

# if working on maps at zenith for each detector
# resfit, mapxy, mapfit, extent, fig, ijres, ijerr = pmf.fit_one_tes(allmaps[idx, :], xs, reso, rot=myrotinit, return_images=True, renorm=True, doplot=True, azelguess=zenith, verbose=True, distok=25)

plt.show()
# sys.exit()

In [ ]:
print(resfit.values)

## Now the fit on all TES

In [ ]:
# for now

visibly_ok_arr = np.ones(len(allTESNum), dtype=bool)

In [ ]:
# %%script echo skipping

# We then perform it for all TES and store the results

# print(allTESNum)
# print(np.shape(allmaps))

reso = 6 #4 #5
xs = 301
distok = 25 # pixels!
allamp = np.zeros(len(allTESNum))
allerramp = np.zeros(len(allTESNum))
allFWHM = np.zeros(len(allTESNum)) * np.nan
allerrFWHM = np.zeros(len(allTESNum))
allpos_zen = np.zeros((len(allTESNum), 2)) * np.nan
allerr_zen = np.zeros((len(allTESNum), 2))

moon_fit = []

for i in range(len(allTESNum)):
    print(allTESNum[i])
    resfit, ijres, ijerr = pmf.fit_one_tes(remap(allmaps[i,:]), xs, reso, rot=zenith, azelguess=fitted_maynooth[i], verbose=False, renorm=True, doplot=False, distok=distok)
    # if we work on maps at zenith for each detector
    # resfit, ijres, ijerr = pmf.fit_one_tes(allmaps[i,:], xs, reso, rot=zenith, azelguess=zenith, verbose=False, renorm=True, doplot=False, distok=distok)
    allFWHM[i] = resfit.values[3] * pmf.conv_reso_fwhm
    allerrFWHM[i] = resfit.errors[3] * pmf.conv_reso_fwhm
    allamp[i] = resfit.values[0] * pmf.conv_reso_fwhm
    allerramp[i] = resfit.errors[0] * pmf.conv_reso_fwhm
    allpos_zen[i, :] = [ijres[1], ijres[0]] # azt, eltc
    allerr_zen[i, :] = [ijerr[1], ijerr[0]]
    print('TES#{0}: FWHM = {1:5.2f}'.format(i + 1, resfit.values[3] * pmf.conv_reso_fwhm))
    moon_fit.append([allpos_zen[i, :], allFWHM[i], allamp[i]])

## We now compare our fitted offsets and the ones from Maynooth

In [ ]:
np.shape(allpos_zen)

In [ ]:
allpos_zen_cart = pmf.spherical2cartesian(1, allpos_zen[:, 0], allpos_zen[:, 1], coord="horizontal", axis="first")
rot_mat = pmf.get_simple_rotation_matrix(axis="y", angle=np.radians(-90)) # we want the det_pos to be aligned with horizon to match Maynooth simulation
allpos_horiz_cart = np.einsum("il,ik->kl", allpos_zen_cart, rot_mat)
# to have the right order in the axes
allpos_zen_cart = np.moveaxis(allpos_zen_cart, source=[0, 1], destination=[1, 0])
allpos_horiz_cart = np.moveaxis(allpos_horiz_cart, source=[0, 1], destination=[1, 0])
#
allpos_horiz = pmf.cartesian2spherical(allpos_horiz_cart[:, 0], allpos_horiz_cart[:, 1], allpos_horiz_cart[:, 2], coord="horizontal", axis="last")[:, 1:]

In [ ]:
print(np.shape(allpos_horiz), np.shape(maynooth_horiz))

In [ ]:
dist_with_maynooth = pmf.dist_angle(maynooth_zen_cart, allpos_zen_cart)
DBscan_ok = pmf.get_DBscan_res(allpos_horiz[:, 0], allpos_horiz[:, 1], maynooth_horiz[:, 0], maynooth_horiz[:, 1], allerr_zen[:, 0], allerr_zen[:, 1], allFWHM, allerrFWHM, visibly_ok_arr, doplot=True, eps=0.2, min_samples=10)

DBscan_ok_cart = pmf.get_DBscan_res_cart(allpos_zen_cart[:, 0], allpos_zen_cart[:, 1], allpos_zen_cart[:, 2], maynooth_zen_cart[:, 0], maynooth_zen_cart[:, 1], maynooth_zen_cart[:, 2], visibly_ok_arr, doplot=True, eps=0.01, min_samples=10)

azte

In [ ]:
# %%script echo offsets calsource
### Offsets from Aoife

dirfiles_ = "/Users/huchet/Documents/code/data/ComissioningTD/calib_lab/Synthesized_Beams_Files/"
fit_pos = np.load(dirfiles_ + "Measured_beam_zeroth_peak.npy")
all_fit_pos_lab = np.zeros((len(allTESNum), 2)) + np.nan
for i in range(len(allTESNum)):
    TESNum = i + 1
    if TESNum in fit_pos[0]:
        all_fit_pos_lab[i] = fit_pos[1:, fit_pos[0] == TESNum].flatten()

######## Here we apply the inversion betwwen Az and El ##############
invert_azel = False # was due to an inversion in the numbering of TES, or was it? It doesn't fit well with the inversion
if invert_azel is True:
    all_fit_pos_lab = np.flip(all_fit_pos_lab, axis=1)
# print(xycreidhe[0])

invert_az = True # might be an error in the data taking (orientation east/west)?
if invert_az is True:
    all_fit_pos_lab[:, 0] = -all_fit_pos_lab[:, 0]
# print(xycreidhe[0])

invert_el = True # might be because we do el - elmoon in our data?
if invert_el is True:
    all_fit_pos_lab[:, 1] = -all_fit_pos_lab[:, 1]

# Compute the offsets centered on zenith
lab_cart = pmf.spherical2cartesian(1, all_fit_pos_lab[:, 0], all_fit_pos_lab[:, 1], coord="horizontal", axis="last") # changed from "first" to "last" in order to be compatible with comes later
rot_mat_zen = pmf.get_simple_rotation_matrix(axis="y", angle=np.radians(90)) # we go back to zenith
lab_zen_cart = np.einsum("li,ik->lk", lab_cart, rot_mat_zen)
lab_zen = pmf.cartesian2spherical(lab_zen_cart[:, 0], lab_zen_cart[:, 1], lab_zen_cart[:, 2], coord="horizontal", axis="last")[:, 1:]

In [ ]:
# %%script echo Aoife offset
DBscan_ok_lab = pmf.get_DBscan_res(allpos_horiz[:, 0], allpos_horiz[:, 1], all_fit_pos_lab[:, 0], all_fit_pos_lab[:, 1], allerr_zen[:, 0], allerr_zen[:, 1], allFWHM, allerrFWHM, visibly_ok_arr, doplot=True, eps=1, min_samples=10)


In [ ]:
# %%script echo skipping

plt.figure()
plt.subplot().set_aspect(1)
# plt.figure()
plt.plot(all_fit_pos_lab[:,  0], all_fit_pos_lab[:,  1], 'bo', alpha=0.2)
plt.plot(all_fit_pos_lab[DBscan_ok_lab,  0], all_fit_pos_lab[DBscan_ok_lab,  1], 'bo', label="Lab positions")
plt.plot(maynooth_horiz[:,  0], maynooth_horiz[:,  1], 'ro', alpha=0.2)
plt.plot(maynooth_horiz[DBscan_ok,  0], maynooth_horiz[DBscan_ok,  1], 'ro', label="Simulated positions")

# plt.plot(allxy[:,0], allxy[:,1], 'ko', label="Moon positions")
plt.plot(allpos_horiz[DBscan_ok, 0], allpos_horiz[DBscan_ok, 1], 'ko', label="Moon positions")
plt.xlabel('$az^{Moon}$')
plt.ylabel('$el^{Moon}$')
plt.legend()
plt.tight_layout()
plt.savefig(dir_plots + "simulated_vs_fitted_moonpos.pdf", dpi=600)
plt.show()

In [ ]:
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(projection='polar')
ax.plot(np.radians(maynooth_zen[:,  0]), 90 - maynooth_zen[:,  1], 'ko', alpha=0.2)
ax.plot(np.radians(maynooth_zen[DBscan_ok,  0]), 90 - maynooth_zen[DBscan_ok,  1], 'ko', label="Simulated positions")
ax.plot(np.radians(allpos_zen[DBscan_ok, 0]), 90 - allpos_zen[DBscan_ok, 1], 'go', label="Observed positions")
ax.plot(np.radians(lab_zen[DBscan_ok_lab, 0]), 90 - lab_zen[DBscan_ok_lab, 1], 'bo', label="Lab positions")

# plt.plot(allxy_rot[:,0], allxy_rot[:,1], 'ko', label="Fitted positions")
ax.set_xlabel('$el^{Moon}$')
ax.set_ylabel('$az^{Moon}$')
ax.legend()
plt.tight_layout()
plt.savefig(dir_plots + "simulated_vs_observed_moonpos_zen.pdf", dpi=600)
plt.show()

In [ ]:
### Now apply it
n_ok = DBscan_ok.sum()

# computing the error
fake_pos = allpos_zen - allerr_zen
fake_pos_cart = pmf.spherical2cartesian(1, fake_pos[:, 0], fake_pos[:, 1], coord="horizontal", axis="last")
all_det_err_cart = np.abs(allpos_zen_cart - fake_pos_cart) * 1

initvec = maynooth_zen_cart[DBscan_ok]

outvec = allpos_zen_cart[DBscan_ok, :]
weightvec = 1./np.sum(all_det_err_cart[DBscan_ok]**2, axis=1)
print(np.shape(initvec))
print(np.shape(outvec))

print(np.shape(weightvec))

initvec_sph = pmf.cartesian2spherical(initvec[:, 0], initvec[:, 1], initvec[:, 2], coord="horizontal", axis="last")[:, 1:]
outvec_sph = pmf.cartesian2spherical(outvec[:, 0], outvec[:, 1], outvec[:, 2], coord="horizontal", axis="last")[:, 1:]

fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(projection='polar')
# fig, ax = plt.subplots(1, 1)
ax.set_aspect(1)
# ax.plot(initvec[:, 0], initvec[:, 1], 'bo', label = 'Créidhe')
# ax.plot(np.radians(initvec_sph[:, 0]), 90 - initvec_sph[:, 1], 'bo', alpha=0.5, label = 'Maynooth')
# ax.plot(outvec[:,0], outvec[:,1], 'ro', label='Moon')
# ax.plot(allpos_zen[isok_arr_dflt, 0], allpos_zen[isok_arr_dflt, 1], 'go', label='Moon VI')
# ax.plot(allpos_zen[DBscan_ok, 0], allpos_zen[DBscan_ok, 1], 'go', label='Moon VI')
# ax.plot(allpos_zen_cart[DBscan_ok, 0], allpos_zen_cart[DBscan_ok, 1], 'go', label='Moon VI')
ax.plot(np.radians(outvec_sph[:, 0]), 90 - outvec_sph[:, 1], 'go', label='Moon ({} TES)'.format(n_ok))

# guess = np.array([0., maz, mel, 1.])
guess = np.array([0, 0, 0])
print(guess)

# # function_fit = pmf.rot_trans_scale_pts
function_fit = pmf.rotate_2d_zen_pts
# # data = fit.Data(np.ravel(initvec), np.ravel(outvec), np.ravel(initvec)*0+1, function_fit)
# data = fit.Data(np.ravel(initvec), np.ravel(outvec), np.ravel(all_det_err_cart[DBscan_ok]), function_fit)
# m_fit_sim_data, ch2, ndf = data.fit_minuit(guess)#, fixpars=[3, 4])

theta_0 = guess
theta_fit = least_squares(pmf.fun_minimise, theta_0, args=(initvec.flatten(), outvec.flatten()))

# newvec = np.reshape(function_fit(np.ravel(initvec), m_fit_sim_data.values), (n_ok, 3))
# newvec = np.reshape(function_fit(np.ravel(initvec), [2.5, -0.5, 2]), (n_ok, 3))
newvec = np.reshape(pmf.rotate_2d_zen_pts(np.ravel(initvec), theta_fit.x), np.shape(initvec))

newvec_sph = pmf.cartesian2spherical(newvec[:, 0], newvec[:, 1], newvec[:, 2], coord="horizontal", axis="last")[:, 1:]

# res = m_fit_sim_data.values
# err = m_fit_sim_data.errors
res = theta_fit.x
err = [theta_fit.cost for i in range(len(res))]

mylabel = 'Fit to match Moon: \n'+ r'Rot$_x$={0:3.2f}+/-{1:3.2f} deg'.format(res[0], err[0])
mylabel += '\n' + r'Rot$_y$={0:3.2f}+/-{1:3.2f} deg'.format(res[1], err[1])
mylabel += '\n' + r'Rot$_z$={0:3.2f}+/-{1:3.2f} deg'.format(res[2], err[2])
# ax.plot(newvec[:,0], newvec[:,1], 'b+', ms=13, label=mylabel)
# ax.plot(np.radians(newvec_sph[:, 0]), 90 - newvec_sph[:, 1], 'b+', ms=13, label=mylabel)
ax.plot(np.radians(newvec_sph[:, 0]), 90 - newvec_sph[:, 1], 'ko', label=mylabel)
ax.legend()
ax.set_xlabel('$\Delta_{az}$ [deg.]')
# ax.set_ylabel('$\Delta_{el}$ [deg.]')
label_position=ax.get_rlabel_position()
ax.text(np.radians(label_position+10), ax.get_rmax()/2., '$\Delta_{el}$ [deg.]',
        rotation=label_position, ha='center', va='center')

for pointi in range(len(initvec[:, 0])):
    # plt.plot([outvec[pointi, 0], newvec[pointi, 0]], [outvec[pointi, 1], newvec[pointi, 1]], c="k")
    plt.plot(np.radians([outvec_sph[pointi, 0], newvec_sph[pointi, 0]]), 90 - np.array([outvec_sph[pointi, 1], newvec_sph[pointi, 1]]), c="r")
plt.savefig(dir_plots + "fitted_maynooth_pos.pdf", dpi=600)
plt.show()

### We can then get an approximation of the positions of the TES that didn't have a good fit using the shifted theoretical position

In [ ]:
not_fitted = ~DBscan_ok
print(np.shape(allpos_zen[not_fitted]))
print(np.shape(m.values))

# now the fitting was performed directly at zenith
newvec_full_cart = np.reshape(function_fit(np.ravel(maynooth_zen_cart), theta_fit.x), np.shape(maynooth_zen_cart))
newvec_full = pmf.cartesian2spherical(newvec_full_cart[:, 0], newvec_full_cart[:, 1], newvec_full_cart[:, 2], coord="horizontal", axis="last")[:, 1:]
print(np.shape(newvec_full))
print(np.shape(newvec_full[not_fitted]))
allpos_zen[not_fitted] = newvec_full[not_fitted]

full_det_pos_zen = allpos_zen.copy()
fitted_maynooth = newvec_full.copy()

np.save('full_det_pos_zen.npy', full_det_pos_zen)
np.save('fitted_maynooth.npy', fitted_maynooth)
print("saved")